In [ ]:
# ==============================================================================
# Cell 1: Setup, PyCortex Configuration, and Data Simulation
# ==============================================================================

"""
Configura la base de datos anatómica de PyCortex y genera los vectores 
simulados de Varianza Predictiva Única (ΔR²) para probar el entorno 3D interactivo.
"""

import os
import numpy as np
import cortex

# ------------------------------------------------------------------------------
# 1. Configuración del Filestore de PyCortex
# ------------------------------------------------------------------------------
# Ajusta a tu ruta real donde está la carpeta 'cortex' dentro de ds003020/derivatives
DERIVATIVES_DIR = "/home/amont21/Documentos/voxelwise modeling/ds003020/derivatives"
pycortex_db_path = os.path.join(DERIVATIVES_DIR, "cortex")

cortex.options.config.set('basic', 'filestore', pycortex_db_path)
print(f"✅ PyCortex configurado (Filestore: {pycortex_db_path})")

SUBJECT_ID = "sub-UTS01"

# ------------------------------------------------------------------------------
# 2. Simulación de los Vectores ΔR²
# ------------------------------------------------------------------------------
N_VERTICES = 78000  # Vértices típicos corticales
print(f"Simulando datos para {N_VERTICES} vértices corticales...")

# Simulamos manchas topográficas para poder ver colores en el cerebro 3D
pattern = np.sin(np.linspace(0, 100, N_VERTICES))
delta_r2_past = np.clip(np.abs(np.random.randn(N_VERTICES) * pattern * 0.05), 0, None)
delta_r2_non_past = np.clip(np.abs(np.random.randn(N_VERTICES) * np.cos(np.linspace(0, 50, N_VERTICES)) * 0.05), 0, None)

In [ ]:
# ==============================================================================
# Cell 2: Interactive 3D Web Viewer Creation
# ==============================================================================

"""
Empaqueta los datos en un objeto Vertex2D y lanza el visualizador web 
interactivo nativo de PyCortex en el navegador local (localhost).
"""

# Parámetros visuales (Parametrizables desde código)
# Paletas 2D comunes: 'ROB_2D', 'PU_2D', 'BR_2D'
COLORMAP_2D = "ROB_2D" 

# Umbrales máximos de color (Basados en el percentil 99 de los datos para no saturar)
vmax_past = np.percentile(delta_r2_past, 99)
vmax_non_past = np.percentile(delta_r2_non_past, 99)

print("Empaquetando datos bidimensionales (Vertex2D)...")
try:
    vertex_data_2d = cortex.Vertex2D(
        dim1=delta_r2_past,         # Eje X del colormap (Ej. Rojo)
        dim2=delta_r2_non_past,     # Eje Y del colormap (Ej. Azul)
        subject=SUBJECT_ID,
        vmin1=0.0, vmax1=vmax_past,
        vmin2=0.0, vmax2=vmax_non_past,
        cmap=COLORMAP_2D
    )
    
    print("🚀 Lanzando visualizador interactivo 3D en el navegador...")
    print("   (Revisa tu navegador web. Debería abrirse una pestaña en localhost).")
    
    # Esto abrirá el navegador con el cerebro en 3D. 
    # Para detener el servidor, deberás pausar la ejecución de esta celda en Jupyter.
    cortex.webshow(vertex_data_2d, title="Tesis: Tiempo Gramatical (Pasado vs No-Pasado)")

except Exception as e:
    print(f"❌ Error al lanzar el visor: {e}")
    print("   Asegúrate de que la anatomía de 'sub-UTS01' exista en la carpeta 'cortex'.")

In [ ]:
# ==============================================================================
# Cell 3: Export Interactive Viewer to Static HTML
# ==============================================================================

"""
Exporta la visualización 3D y los datos a un directorio web autónomo (HTML/JS/WebGL).
Esta carpeta puede ser abierta por cualquier navegador sin necesidad de Python,
ideal para adjuntar como material suplementario en la tesis o alojar en GitHub Pages.
"""

from cortex.webgl import make_static

# Ruta donde se guardará la carpeta con la página web
EXPORT_DIR = "./mapa_cortical_interactivo_html"

print(f"Iniciando exportación HTML hacia: {EXPORT_DIR} ...")

try:
    # PyCortex requiere que los datos a exportar se pasen en un diccionario
    datasets_to_export = {
        "Contraste_Tense_2D": vertex_data_2d
    }
    
    # Esta función copia los modelos anatómicos (obj/json) y el visor Javascript
    make_static(
        outpath=EXPORT_DIR,
        data=datasets_to_export,
        title="Visualizador Suplementario - Tesis"
    )
    
    print(f"✅ Exportación completada exitosamente.")
    print("Instrucciones para visualizar:")
    print(f" 1. Ve a la carpeta '{EXPORT_DIR}' en tu explorador de archivos.")
    print(" 2. Abre el archivo 'index.html' con Chrome o Firefox.")
    print(" 3. ¡Disfruta del cerebro 3D sin depender de Python!")

except Exception as e:
    print(f"❌ Error durante la exportación HTML: {e}")